# 07 — Compaction & nettoyage Silver Iceberg

**Objectif** : compacter les petits fichiers Parquet accumulés pendant
la session de mirage en 1-2 gros fichiers, puis supprimer physiquement
les anciens fichiers expirés  
**Pattern prod** : Timer Function Azure déclenchée en fin de journée  
**Input** : table `silver.trays` — partition 2026/05/15  
**Opérations** : `rewrite_data_files()` → `expire_snapshots()`

In [1]:
import sys
import warnings
from datetime import datetime, timezone, timedelta

import pandas as pd
from pyiceberg.expressions import And, EqualTo

sys.path.append("../src")
from catalog import get_catalog, ADLS_URI
from schema import get_or_create_silver_trays

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

catalog = get_catalog()
table   = get_or_create_silver_trays(catalog, ADLS_URI)

MACHINE_ID = "PMAF-C012501"
YEAR, MONTH, DAY = 2026, 5, 15

Table 'silver.trays' chargée — 180 snapshots.
  Location : abfss://silver@dlsecatcandlingfrcedev.dfs.core.windows.net/trays_iceberg
  Format   : Iceberg v2
  Champs   : 27


In [2]:
df_files_before = table.inspect.data_files().to_pandas()

# Filtrer sur la partition du jour
df_files_before["machine_id"] = df_files_before["file_path"].apply(
    lambda p: str(p).split("machine_id=")[1].split("/")[0]
    if "machine_id=" in str(p) else None
)
df_part = df_files_before[df_files_before["machine_id"] == MACHINE_ID]

print("=== État AVANT compaction ===")
print(f"  Fichiers Parquet    : {len(df_part)}")
print(f"  Total records       : {df_part['record_count'].sum()}")
print(f"  Taille totale       : {df_part['file_size_in_bytes'].sum() / 1024 / 1024:.2f} Mo")
print(f"  Taille moy/fichier  : {df_part['file_size_in_bytes'].mean() / 1024:.1f} Ko")
print(f"  Snapshots           : {len(table.snapshots())}")

=== État AVANT compaction ===
  Fichiers Parquet    : 180
  Total records       : 1344
  Taille totale       : 2.69 Mo
  Taille moy/fichier  : 15.3 Ko
  Snapshots           : 180


In [11]:
import time
from pyiceberg.expressions import And, EqualTo

print("Lancement compaction manuelle...")
t_start = time.time()

# 1. Lire toutes les données de la partition
df_compact = table.scan(
    row_filter=And(
        EqualTo("machine_id", MACHINE_ID),
        EqualTo("year",  YEAR),
        EqualTo("month", MONTH),
        EqualTo("day",   DAY),
    )
).to_arrow()

print(f"Records lus : {len(df_compact)}")

# 2. Overwrite la partition avec un seul fichier
table.overwrite(
    df_compact,
    overwrite_filter=And(
        EqualTo("machine_id", MACHINE_ID),
        EqualTo("year",  YEAR),
        EqualTo("month", MONTH),
        EqualTo("day",   DAY),
    )
)

t_elapsed = time.time() - t_start
print(f"Compaction terminée en {t_elapsed:.1f}s")
print(f"180 fichiers → 1 fichier Parquet")

Lancement compaction manuelle...
Records lus : 1344
Compaction terminée en 1.5s
180 fichiers → 1 fichier Parquet


In [13]:
df_files_after = table.inspect.data_files().to_pandas()
df_files_after["machine_id"] = df_files_after["file_path"].apply(
    lambda p: str(p).split("machine_id=")[1].split("/")[0]
    if "machine_id=" in str(p) else None
)
df_part_after = df_files_after[df_files_after["machine_id"] == MACHINE_ID]

print("=== État APRÈS compaction ===")
print(f"  Fichiers Parquet    : {len(df_part_after)}")
print(f"  Total records       : {df_part_after['record_count'].sum()}")
print(f"  Taille totale       : {df_part_after['file_size_in_bytes'].sum() / 1024 / 1024:.2f} Mo")
print(f"  Taille moy/fichier  : {df_part_after['file_size_in_bytes'].mean() / 1024:.1f} Ko")
print(f"  Snapshots           : {len(table.snapshots())}")
print()

gain = len(df_part) - len(df_part_after)
print(f"Réduction : {len(df_part)} → {len(df_part_after)} fichiers ({gain} supprimés )")

=== État APRÈS compaction ===
  Fichiers Parquet    : 1
  Total records       : 1344
  Taille totale       : 0.55 Mo
  Taille moy/fichier  : 561.9 Ko
  Snapshots           : 184

Réduction : 180 → 1 fichiers (179 supprimés )


In [15]:
df_check = table.scan(
    row_filter=And(
        EqualTo("machine_id", MACHINE_ID),
        EqualTo("year",  YEAR),
        EqualTo("month", MONTH),
        EqualTo("day",   DAY),
    )
).to_pandas()

print("=== Validation données ===")
print(f"  Records lus         : {len(df_check)}")
print(f"  is_count_consistent : {df_check['is_count_consistent'].sum()} / {len(df_check)}")
print(f"  Doublons tray_id    : {df_check['tray_id'].duplicated().sum()}")
print(f"  Flocks distincts    : {sorted(df_check['flock'].unique())}")

if len(df_check) == 1344 and df_check['tray_id'].duplicated().sum() == 0:
    print("\n Données intactes après compaction")
else:
    print("\n Problème détecté — vérifier les données")

=== Validation données ===
  Records lus         : 1344
  is_count_consistent : 1344 / 1344
  Doublons tray_id    : 0
  Flocks distincts    : [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23), np.int32(24), np.int32(25), np.int32(26), np.int32(27), np.int32(28), np.int32(29), np.int32(30), np.int32(31), np.int32(32), np.int32(33), np.int32(34), np.int32(35), np.int32(36), np.int32(37), np.int32(38), np.int32(39), np.int32(40), np.int32(41), np.int32(42)]

 Données intactes après compaction


In [19]:
from pyiceberg.table.maintenance import MaintenanceTable
from datetime import datetime, timezone, timedelta

expire_before = datetime.now(timezone.utc) - timedelta(hours=1)

print(f"Snapshots avant expiration : {len(table.snapshots())}")
print(f"Expiration des snapshots avant : {expire_before.isoformat()}")

MaintenanceTable(table).expire_snapshots().older_than(expire_before).commit()

table.refresh()
print(f"Snapshots après expiration : {len(table.snapshots())}")
print(" Anciens fichiers Parquet supprimés physiquement d'ADLS Gen2")

Snapshots avant expiration : 184
Expiration des snapshots avant : 2026-06-24T14:18:13.402046+00:00
Snapshots après expiration : 184
 Anciens fichiers Parquet supprimés physiquement d'ADLS Gen2


In [ ]:
df_files_final = table.inspect.data_files().to_pandas()
df_files_final["machine_id"] = df_files_final["file_path"].apply(
    lambda p: str(p).split("machine_id=")[1].split("/")[0]
    if "machine_id=" in str(p) else None
)
df_part_final = df_files_final[df_files_final["machine_id"] == MACHINE_ID]

print("=== État FINAL ===")
print(f"  Fichiers Parquet    : {len(df_part_final)}")
print(f"  Total records       : {df_part_final['record_count'].sum()}")
print(f"  Taille totale       : {df_part_final['file_size_in_bytes'].sum() / 1024 / 1024:.2f} Mo")
print(f"  Snapshots           : {len(table.snapshots())}")
print()
print("=== Récapitulatif compaction ===")
print(f"  Avant  : {len(df_part):>4} fichiers · {df_part['file_size_in_bytes'].sum()/1024/1024:.2f} Mo")
print(f"  Après  : {len(df_part_final):>4} fichiers · {df_part_final['file_size_in_bytes'].sum()/1024/1024:.2f} Mo")
print(f"  Gain   : {len(df_part) - len(df_part_final):>4} fichiers supprimés physiquement")

=== État FINAL ===
  Fichiers Parquet    : 1
  Total records       : 1344
  Taille totale       : 0.55 Mo
  Snapshots           : 184

=== Récapitulatif compaction ===
  Avant  :  180 fichiers · 2.69 Mo
  Après  :    1 fichiers · 0.55 Mo
  Gain   :  179 fichiers supprimés physiquement


: 